In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
restaurantes_clean = spark.table(
    "workspace.default.restaurantes_clean"
)

reviews_clean = spark.table(
    "workspace.default.reviews_clean"
)

model_clean = spark.table(
    "workspace.default.model_clean"
)

**Crear dim_restaurant**

Nuestra dimensión de restaurantes tendrá información descriptiva del establecimiento.

Primero crearemos una versión con las columnas limpias.

In [0]:
dim_restaurant = (
    restaurantes_clean
    .select(
        F.col("id_clean").alias("restaurant_id"),
        F.trim(F.col("name")).alias("restaurant_name"),
        F.trim(F.col("tag")).alias("tags"),
        F.col("x_clean").alias("latitude"),
        F.col("y_clean").alias("longitude"),
        F.upper(F.trim(F.col("district"))).alias("district"),
        F.trim(F.col("IDDIST")).alias("district_id"),
        F.trim(F.col("direction")).alias("address"),
        F.col("stars_clean").alias("restaurant_stars"),
        F.col("n_reviews_clean").alias("restaurant_review_count"),
        F.col("min_price_clean").alias("min_price"),
        F.col("max_price_clean").alias("max_price"),
        F.trim(F.col("platform")).alias("restaurant_platform")
    )
    .filter(F.col("restaurant_id").isNotNull())
    .dropDuplicates(["restaurant_id"])
)

Ahora añadimos dos variables útiles para el análisis:

In [0]:
dim_restaurant = (
    dim_restaurant
    .withColumn(
        "average_price",
        F.when(
            F.col("min_price").isNotNull() &
            F.col("max_price").isNotNull(),
            (F.col("min_price") + F.col("max_price")) / 2
        )
    )
    .withColumn(
        "price_range",
        F.when(F.col("average_price").isNull(), "Unknown")
         .when(F.col("average_price") < 50, "Low")
         .when(F.col("average_price") < 100, "Medium")
         .when(F.col("average_price") < 200, "High")
         .otherwise("Premium")
    )
)

In [0]:
print("Cantidad de restaurantes:", dim_restaurant.count())

display(dim_restaurant.limit(10))

**Crear fact_reviews**

Esta será nuestra tabla principal.

In [0]:
fact_reviews = (
    reviews_clean
    .select(
        F.trim(F.col("id_review")).alias("review_id"),
        F.col("restaurant_id").alias("restaurant_id"),
        F.trim(F.col("review")).alias("review_text"),
        F.trim(F.col("title")).alias("review_title"),
        F.col("score_clean").alias("score"),
        F.col("likes_clean").alias("likes"),
        F.trim(F.col("id_nick")).alias("user_nickname"),
        F.trim(F.col("date")).alias("review_date_text"),
        F.trim(F.col("platform")).alias("review_platform")
    )
    .filter(
        F.col("review_id").isNotNull() &
        F.col("restaurant_id").isNotNull()
    )
)

Ahora añadiremos categorías de satisfacción:

In [0]:
fact_reviews = (
    fact_reviews
    .withColumn(
        "satisfaction_level",
        F.when(F.col("score") <= 2, "Low")
         .when(F.col("score") == 3, "Neutral")
         .otherwise("High")
    )
)

In [0]:
display(
    fact_reviews
    .groupBy("satisfaction_level")
    .count()
)

**Integrar las emociones del modelo**

Ahora uniremos:

fact_reviews
      +
model_clean

In [0]:
model_features = (
    model_clean
    .select(
        F.trim(F.col("id_review")).alias("review_id"),
        "anger",
        "anticipation",
        "disgust",
        "fear",
        "joy",
        "sadness",
        "surprise",
        "trust",
        "negative",
        "positive",
        "stars_1",
        "stars_2",
        "stars_3",
        "stars_4",
        "stars_5"
    )
)

¿Por qué usamos left?

Porque no queremos perder las reviews válidas que no aparecen en model.csv.

In [0]:
fact_reviews = (
    fact_reviews
    .join(
        model_features,
        on="review_id",
        how="left"
    )
)

**Crear una variable de sentimiento**

Vamos a aprovechar positive y negative.

In [0]:
fact_reviews = (
    fact_reviews
    .withColumn(
        "sentiment",
        F.when(
            F.col("positive").isNull() |
            F.col("negative").isNull(),
            "Not available"
        )
        .when(F.col("positive") > F.col("negative"), "Positive")
        .when(F.col("negative") > F.col("positive"), "Negative")
        .otherwise("Neutral")
    )
)

In [0]:
display(
    fact_reviews
    .groupBy("sentiment")
    .count()
)

**Identificar la emoción predominante**

Esto hará nuestro proyecto más interesante.

In [0]:
emotion_columns = [
    "anger",
    "anticipation",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "trust"
]

fact_reviews = (
    fact_reviews
    .withColumn(
        "max_emotion_score",
        F.greatest(
            F.col("anger"),
            F.col("anticipation"),
            F.col("disgust"),
            F.col("fear"),
            F.col("joy"),
            F.col("sadness"),
            F.col("surprise"),
            F.col("trust")
        )
    )
    .withColumn(
        "dominant_emotion",
        F.when(F.col("max_emotion_score").isNull(), "Not available")
        .when(F.col("anger") == F.col("max_emotion_score"), "anger")
        .when(F.col("anticipation") == F.col("max_emotion_score"), "anticipation")
        .when(F.col("disgust") == F.col("max_emotion_score"), "disgust")
        .when(F.col("fear") == F.col("max_emotion_score"), "fear")
        .when(F.col("joy") == F.col("max_emotion_score"), "joy")
        .when(F.col("sadness") == F.col("max_emotion_score"), "sadness")
        .when(F.col("surprise") == F.col("max_emotion_score"), "surprise")
        .when(F.col("trust") == F.col("max_emotion_score"), "trust")
        .otherwise("Not available")
    )
    .drop("max_emotion_score")
)

In [0]:
display(
    fact_reviews
    .groupBy("dominant_emotion")
    .count()
    .orderBy(F.desc("count"))
)

**19. Verificar integridad referencial**

Ahora comprobamos si las reviews limpias tienen restaurante asociado.

In [0]:
reviews_without_restaurant = (
    fact_reviews
    .join(
        dim_restaurant.select("restaurant_id"),
        on="restaurant_id",
        how="left_anti"
    )
)

print(
    "Reviews válidas sin restaurante:",
    reviews_without_restaurant.count()
)

In [0]:
print("Total fact reviews:", fact_reviews.count())

print(
    "Reviews con información de sentimiento:",
    fact_reviews.filter(
        F.col("positive").isNotNull()
    ).count()
)

print(
    "Reviews sin información de sentimiento:",
    fact_reviews.filter(
        F.col("positive").isNull()
    ).count()
)

**PASO 20 — Guardar las tablas finales**

Primero la dimensión:

In [0]:
dim_restaurant.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_restaurant")

In [0]:
fact_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.fact_reviews")

In [0]:
print("DIM_RESTAURANT:", spark.table(
    "workspace.default.dim_restaurant"
).count())

print("FACT_REVIEWS:", spark.table(
    "workspace.default.fact_reviews"
).count())

In [0]:
reviews_without_restaurant = (
    fact_reviews
    .join(
        dim_restaurant.select("restaurant_id"),
        on="restaurant_id",
        how="left_anti"
    )
)

print(
    "Reviews válidas sin restaurante:",
    reviews_without_restaurant.count()
)

In [0]:
print("Total fact reviews:", fact_reviews.count())

print(
    "Reviews con información de sentimiento:",
    fact_reviews.filter(
        F.col("positive").isNotNull()
    ).count()
)

print(
    "Reviews sin información de sentimiento:",
    fact_reviews.filter(
        F.col("positive").isNull()
    ).count()
)

In [0]:
fact_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.fact_reviews")